# DAM Prices — Extracción y limpieza

**Propósito:** Extraer los precios DAM horarios de España y Polonia, homogeneizar
granularidades (ES/PT liquida a 15 min desde oct-2025, PL es horario nativo)
y producir un DataFrame horario limpio listo para cruzar con Greenbyte.

---
### Estructura del notebook
| Sección | Descripción |
|---|---|
| 0 | Configuración — credenciales, ventana temporal |
| 1 | Extracción desde Azure SQL |
| 2 | Inspección de granularidad real por país y periodo |
| 3 | Homogeneización → DataFrame horario único |
| 4 | Guardado en `.pkl` |
| 5 | Validación — carga del pkl + heatmap de horas negativas |


## 0 · Configuración

In [2]:
import pandas as pd
import numpy as np
import pickle
import pyodbc
import warnings
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
from datetime import timezone

warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")

# ── Azure SQL ─────────────────────────────────────────────────────────────────
DB_SERVER   = "exusbigcloud.database.windows.net"
DB_DATABASE = "bigcloud_db"
DB_USERNAME = "exusbigcloud_admin"
DB_PASSWORD = "7t>Oj038U&%s"

# ── Ventana temporal ──────────────────────────────────────────────────────────
DATE_START = "2025-01-01"
DATE_END   = "2026-03-31"

# ── Países relevantes para nuestros parques ───────────────────────────────────
#   ES → Cascante
#   PL → Okonek, Jastrowie
COUNTRIES = ["ES", "PT", "PL"]

# ── Fecha de cambio a granularidad 15min en MIBEL (ES/PT) ────────────────────
MIBEL_15MIN_START = pd.Timestamp("2025-10-01", tz="UTC")

# ── Colores Exus ──────────────────────────────────────────────────────────────
EXUS_DARK_BLUE  = "#002060"
EXUS_MID_BLUE   = "#0070C0"
EXUS_LIGHT_BLUE = "#BDD7EE"
EXUS_ORANGE     = "#FF6600"
EXUS_GRAY       = "#595959"

print("Configuración cargada ✓")
print(f"Ventana: {DATE_START} → {DATE_END}")
print(f"Países: {COUNTRIES}")


Configuración cargada ✓
Ventana: 2025-01-01 → 2026-03-31
Países: ['ES', 'PT', 'PL']


## 1 · Extracción desde Azure SQL

Descargamos `EuropeDAMActuals` para ES y PL en el periodo 2025-2026.
Pedimos todos los registros sin asumir granularidad — la inspeccionamos en la
sección siguiente.


In [3]:
conn_str = (
    f"DRIVER={{SQL Server}};"
    f"SERVER={DB_SERVER};DATABASE={DB_DATABASE};"
    f"UID={DB_USERNAME};PWD={DB_PASSWORD}"
)
conn = pyodbc.connect(conn_str)
print("Conexión establecida ✓")


Conexión establecida ✓


In [4]:
query = f"""
SELECT CountryCode, UTCTime, Price
FROM [dbo].[EuropeDAMActuals]
WHERE Market = 'DAM'
  AND CountryCode IN ('ES', 'PT', 'PL')
  AND UTCTime >= '{DATE_START}'
  AND UTCTime <  '{DATE_END} 23:59:59'
ORDER BY CountryCode, UTCTime
"""

df_raw = pd.read_sql(query, conn)
conn.close()

print(f"Filas descargadas: {len(df_raw):,}")
print(f"Países: {df_raw['CountryCode'].unique()}")
print()
print(df_raw.head(6).to_string(index=False))


Filas descargadas: 74,298
Países: ['ES' 'PL' 'PT']

CountryCode                   UTCTime  Price
         ES 2025-01-01T00:00:00+00:00 131.59
         ES 2025-01-01T01:00:00+00:00 131.49
         ES 2025-01-01T02:00:00+00:00 131.42
         ES 2025-01-01T03:00:00+00:00 120.49
         ES 2025-01-01T04:00:00+00:00 112.30
         ES 2025-01-01T05:00:00+00:00 112.30


## 2 · Inspección de granularidad real

Verificamos qué resolución tiene realmente la BD para cada país y periodo,
calculando el intervalo modal entre timestamps consecutivos.


In [5]:
# ── Parsear timestamps ────────────────────────────────────────────────────────
df_raw["timestamp"] = pd.to_datetime(
    df_raw["UTCTime"].astype(str).str.replace(" ", "T"), utc=True
)
df_raw = df_raw.drop(columns=["UTCTime"]).sort_values(["CountryCode", "timestamp"])

# ── Intervalo modal por país y tramo temporal ─────────────────────────────────
print("Granularidad por país y tramo:")
print(f"{'País':<6} {'Tramo':<25} {'Intervalo modal':<20} {'N registros':>12}")
print("─" * 68)

for cc in COUNTRIES:
    sub = df_raw[df_raw["CountryCode"] == cc].copy()

    # Antes del cambio 15min
    before = sub[sub["timestamp"] < MIBEL_15MIN_START]
    after  = sub[sub["timestamp"] >= MIBEL_15MIN_START]

    for label, chunk in [("antes oct-2025", before), ("desde oct-2025", after)]:
        if chunk.empty:
            continue
        diffs = chunk["timestamp"].diff().dropna()
        modal = diffs.mode()[0] if not diffs.empty else "N/A"
        print(f"  {cc:<4} {label:<25} {str(modal):<20} {len(chunk):>12,}")


Granularidad por país y tramo:
País   Tramo                     Intervalo modal       N registros
────────────────────────────────────────────────────────────────────
  ES   antes oct-2025            0 days 01:00:00             9,054
  ES   desde oct-2025            0 days 00:15:00            17,376
  PT   antes oct-2025            0 days 01:00:00             6,558
  PT   desde oct-2025            0 days 00:15:00            17,376
  PL   antes oct-2025            0 days 01:00:00             6,558
  PL   desde oct-2025            0 days 00:15:00            17,376


In [6]:
# ── Cuántos periodos por hora para ES y PT desde oct-2025 (mercado MIBEL) ────
for cc in ["ES", "PT"]:
    cc_after = df_raw[
        (df_raw["CountryCode"] == cc) &
        (df_raw["timestamp"] >= MIBEL_15MIN_START)
    ].copy()

    if cc_after.empty:
        print(f"{cc}: sin datos desde oct-2025\n")
        continue

    cc_after["hour_floor"] = cc_after["timestamp"].dt.floor("h")
    periods_per_hour = cc_after.groupby("hour_floor").size()

    print(f"Distribución de periodos por hora ({cc} desde oct-2025):")
    print(periods_per_hour.value_counts().sort_index().rename("n_horas"))
    print(f"Horas con exactamente 4 periodos: {(periods_per_hour == 4).sum():,} "
          f"({(periods_per_hour == 4).mean()*100:.1f}%)")
    print(f"Horas con ≠ 4 periodos:           {(periods_per_hour != 4).sum():,}")
    print()

Distribución de periodos por hora (ES desde oct-2025):
4    4344
Name: n_horas, dtype: int64
Horas con exactamente 4 periodos: 4,344 (100.0%)
Horas con ≠ 4 periodos:           0

Distribución de periodos por hora (PT desde oct-2025):
4    4344
Name: n_horas, dtype: int64
Horas con exactamente 4 periodos: 4,344 (100.0%)
Horas con ≠ 4 periodos:           0



## 3 · Homogeneización → DataFrame horario único

**Estrategia:**
- **PL** (siempre horario): sin transformación, se usa directamente.
- **ES antes de oct-2025** (horario): sin transformación.
- **ES desde oct-2025** (15 min → 1h): promedio simple de los 4 periodos de cada hora.
  El promedio simple es equivalente al promedio ponderado cuando los 4 periodos tienen
  el mismo volumen teórico disponible (que es el caso en el DAM).

El resultado es un único DataFrame con columnas:
`timestamp_utc` (inicio de hora, UTC) · `country` · `price_eur_mwh` · `n_periods` · `has_negative_period`

La columna `has_negative_period` indica si alguno de los periodos de 15min
que componen esa hora fue negativo — útil como flag de calidad aunque la
hora media sea positiva.


In [7]:
def homogenize_to_hourly(df: pd.DataFrame) -> pd.DataFrame:
    """
    Convierte el DataFrame raw (mezcla de granularidades) a resolución horaria.

    Para cada país:
      - Agrupa por hora (floor al inicio de hora UTC)
      - Calcula precio medio horario (promedio de los N periodos)
      - Registra n_periods (cuántos periodos había: 1 si era horario, 4 si 15min)
      - Registra has_negative_period (si algún periodo individual era < 0)

    Parámetros
    ----------
    df : DataFrame con columnas [CountryCode, timestamp, Price]

    Retorna
    -------
    DataFrame con columnas:
        timestamp_utc, country, price_eur_mwh, n_periods, has_negative_period
    """
    df = df.copy()
    df["hour"] = df["timestamp"].dt.floor("h")

    agg = (
        df.groupby(["CountryCode", "hour"])
        .agg(
            price_eur_mwh        = ("Price",  "mean"),
            n_periods            = ("Price",  "count"),
            has_negative_period  = ("Price",  lambda x: (x < 0).any()),
        )
        .reset_index()
        .rename(columns={"CountryCode": "country", "hour": "timestamp_utc"})
    )

    return agg.sort_values(["country", "timestamp_utc"]).reset_index(drop=True)


df_hourly = homogenize_to_hourly(df_raw)

print(f"DataFrame horario: {len(df_hourly):,} filas")
print()
print(df_hourly.groupby("country").agg(
    n_horas      = ("timestamp_utc",    "count"),
    desde        = ("timestamp_utc",    "min"),
    hasta        = ("timestamp_utc",    "max"),
    avg_price    = ("price_eur_mwh",    "mean"),
    n_negativas  = ("price_eur_mwh",    lambda x: (x < 0).sum()),
    n_neg_period = ("has_negative_period", "sum"),
).round(2).to_string())


DataFrame horario: 32,688 filas

         n_horas                     desde                     hasta  avg_price  n_negativas  n_neg_period
country                                                                                                   
ES         10896 2025-01-01 00:00:00+00:00 2026-03-30 23:00:00+00:00      61.19          852           914
PL         10896 2025-01-01 00:00:00+00:00 2026-03-30 23:00:00+00:00     107.64          362           392
PT         10896 2025-01-01 00:00:00+00:00 2026-03-30 23:00:00+00:00      61.49          427           439


In [8]:
# ── Verificación de continuidad horaria ──────────────────────────────────────
print("Huecos en la serie horaria por país:")
for cc in COUNTRIES:
    sub  = df_hourly[df_hourly["country"] == cc]["timestamp_utc"].sort_values()
    gaps = sub.diff().dropna()
    bad  = gaps[gaps > pd.Timedelta("1h")]
    if bad.empty:
        print(f"  {cc}: sin huecos ✓")
    else:
        print(f"  {cc}: {len(bad)} huecos detectados")
        # Mostrar los 5 mayores
        top = bad.nlargest(5)
        for ts, gap in top.items():
            print(f"       {sub.loc[ts - 1] if ts-1 in sub.index else '?'}  →  gap {gap}")

# ── Muestra final ─────────────────────────────────────────────────────────────
print()
print("Muestra (5 filas):")
print(df_hourly.sample(5, random_state=42).to_string(index=False))


Huecos en la serie horaria por país:
  ES: sin huecos ✓
  PT: sin huecos ✓
  PL: sin huecos ✓

Muestra (5 filas):
country             timestamp_utc  price_eur_mwh  n_periods  has_negative_period
     PL 2026-02-19 12:00:00+00:00        107.400          4                False
     ES 2025-06-07 12:00:00+00:00         -0.600          1                 True
     PT 2025-02-05 12:00:00+00:00         35.200          1                False
     ES 2026-02-01 05:00:00+00:00          1.150          4                False
     ES 2026-02-02 08:00:00+00:00         31.005          4                False


## 4 · Guardado en `.pkl`

In [10]:
from datetime import datetime
from pathlib import Path

run_date = datetime.now(timezone.utc).strftime("%Y%m%d")
pkl_path =f"dam_prices_hourly_{run_date}.pkl"

# Guardamos solo los DataFrames de datos — la validación se genera al leer
results = {
    "hourly": df_hourly,   # DataFrame principal — una fila por hora y país
    "raw":    df_raw,      # Datos crudos de la BD (15min de ES desde oct-2025)
}

with open(pkl_path, "wb") as f:
    pickle.dump(results, f, protocol=pickle.HIGHEST_PROTOCOL)

print()
print("Contenido del pkl:")
for k, v in results.items():
    print(f"  results['{k}'] → {type(v).__name__} {v.shape}")
print()
print("Continúa en Sección 5 para la validación →")



Contenido del pkl:
  results['hourly'] → DataFrame (32688, 5)
  results['raw'] → DataFrame (74298, 3)

Continúa en Sección 5 para la validación →


## 5 · Validación — carga del pkl + heatmap de horas negativas

Cargamos el pkl recién guardado y construimos el heatmap desde cero,
verificando que la serialización es correcta y el fichero es autosuficiente.

**Hora negativa** = precio medio horario < 0 €/MWh.


In [ ]:
# ── Cargar pkl ────────────────────────────────────────────────────────────────
with open(pkl_path, "rb") as f:
    loaded = pickle.load(f)

df_val = loaded["hourly"]
print(f"pkl cargado ✓  —  {len(df_val):,} filas")
print()
print(df_val.dtypes)


In [ ]:
# ── Matriz mes × país de horas negativas ─────────────────────────────────────
neg = df_val[df_val["price_eur_mwh"] < 0].copy()
neg["month"] = neg["timestamp_utc"].dt.to_period("M")

pivot = (
    neg.groupby(["month", "country"])
    .size()
    .unstack("country")
    .fillna(0)
    .astype(int)
)

# Rellenar todos los meses del rango aunque no haya negativos
all_months = pd.period_range(DATE_START, DATE_END, freq="M")
pivot = pivot.reindex(all_months, fill_value=0)

print("Horas negativas por mes y país:")
print(pivot.to_string())
print()
for cc in pivot.columns:
    print(f"  Total horas negativas {cc}: {pivot[cc].sum()}")


In [ ]:
# ── Heatmap ───────────────────────────────────────────────────────────────────
data   = pivot.values.astype(float)
months = [str(m) for m in pivot.index]
cols   = list(pivot.columns)

fig, ax = plt.subplots(figsize=(max(6, len(cols) * 2.5), len(months) * 0.55 + 1.5))

cmap = mcolors.LinearSegmentedColormap.from_list(
    "exus_blue",
    ["#FFFFFF", EXUS_LIGHT_BLUE, EXUS_MID_BLUE, EXUS_DARK_BLUE]
)
im = ax.imshow(data, aspect="auto", cmap=cmap, vmin=0, vmax=max(data.max(), 1))

# Ejes — países arriba
ax.set_xticks(range(len(cols)))
ax.set_xticklabels(cols, fontsize=11, fontweight="bold", color=EXUS_DARK_BLUE)
ax.set_yticks(range(len(months)))
ax.set_yticklabels(months, fontsize=9, color=EXUS_GRAY)
ax.xaxis.set_ticks_position("top")
ax.xaxis.set_label_position("top")

# Valores en cada celda
for i in range(len(months)):
    for j in range(len(cols)):
        val        = int(data[i, j])
        brightness = data[i, j] / max(data.max(), 1)
        txt_color  = "white" if brightness > 0.55 else EXUS_DARK_BLUE
        ax.text(j, i, str(val) if val > 0 else "\u2014",
                ha="center", va="center",
                fontsize=10, fontweight="bold", color=txt_color)

# Línea naranja: inicio mercado 15min ES (oct-2025)
oct25_idx = next((i for i, m in enumerate(pivot.index) if str(m) == "2025-10"), None)
if oct25_idx is not None and "ES" in cols:
    ax.axhline(oct25_idx - 0.5, color=EXUS_ORANGE, linewidth=1.5,
               linestyle="--", alpha=0.8)
    ax.text(cols.index("ES"), oct25_idx - 0.72,
            "ES: inicio mercado 15 min",
            fontsize=7.5, color=EXUS_ORANGE, ha="center", fontweight="bold")

# Colorbar
cbar = fig.colorbar(im, ax=ax, shrink=0.6, pad=0.02)
cbar.set_label("Horas negativas / mes", fontsize=9, color=EXUS_GRAY)
cbar.ax.tick_params(colors=EXUS_GRAY)

ax.set_title(
    "Horas de precio negativo DAM por mes y pa\u00eds\n"
    "(precio medio horario < 0 \u20ac/MWh)",
    fontsize=12, fontweight="bold", color=EXUS_DARK_BLUE, pad=18
)
fig.text(
    0.5, -0.01,
    "ES desde oct-2025: precio medio de los 4 periodos de 15 min  "
    "|  PL: precio horario nativo",
    ha="center", fontsize=8, color=EXUS_GRAY, style="italic"
)

plt.tight_layout()
OUTPUT_DIR.mkdir(exist_ok=True)
plt.savefig(OUTPUT_DIR / "negative_hours_heatmap.png", dpi=150,
            bbox_inches="tight", facecolor="white")
plt.show()
print("Heatmap guardado \u2713")


In [ ]:
# ── Estadísticas complementarias ──────────────────────────────────────────────

# Horas con media >= 0 pero algún periodo de 15min negativo (ES desde oct-2025)
mixed = df_val[
    (df_val["country"] == "ES") &
    (df_val["price_eur_mwh"] >= 0) &
    (df_val["has_negative_period"] == True)
]
print(f"ES \u2014 horas con media \u2265 0 pero alg\u00fan periodo de 15min negativo: {len(mixed):,}")
print("(No se cuentan como negativas en nuestra definici\u00f3n)")
print()

# Precio mínimo por país
print("Precio horario m\u00ednimo registrado:")
for cc in COUNTRIES:
    sub = df_val[df_val["country"] == cc]
    row = sub.loc[sub["price_eur_mwh"].idxmin()]
    print(f"  {cc}: {row['price_eur_mwh']:.2f} \u20ac/MWh  \u2014  {row['timestamp_utc']}")

# Distribución de precios negativos
print()
print("Distribuci\u00f3n de precios en horas negativas (\u20ac/MWh):")
for cc in COUNTRIES:
    sub = df_val[(df_val["country"] == cc) & (df_val["price_eur_mwh"] < 0)]
    if sub.empty:
        print(f"  {cc}: sin horas negativas")
        continue
    p = sub["price_eur_mwh"]
    print(f"  {cc}:  n={len(sub):4d}  "
          f"media={p.mean():.1f}  min={p.min():.1f}  "
          f"p25={p.quantile(0.25):.1f}  p75={p.quantile(0.75):.1f}")

# Control de calidad: n_periods
print()
print("n_periods por pa\u00eds (debe ser 1 antes de oct-2025 y 4 despu\u00e9s para ES):")
print(df_val.groupby(["country", "n_periods"]).size().rename("n_horas").to_string())

print()
print("Para usar en Economic_Curtailment:")
print(f"  with open('{pkl_path}', 'rb') as f:")
print(f"      prices = pickle.load(f)")
print(f"  df_prices = prices['hourly']")
